## Summary

- Analyzed 999 Google Reviews across Minimalix, Livspace, and HomeLane.
- Processed 546 text reviews using NLP techniques.
- Identified major satisfaction drivers and complaint themes.
- Found that dissatisfied customers write 2.6x longer reviews.
- HomeLane complaints focus on execution and delays.
- Livspace complaints focus on post-sales service.
- Minimalix complaints focus on lead quality and subscription ROI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')
from google.colab import files

### Data Collection & Performing EDA

In [ ]:
#importing the data from Github
url = 'https://raw.githubusercontent.com/TusharIITg01/voice-of-customer-analytics/main/Company%20Reviews%20%20-%20Sheet1.csv'
df = pd.read_csv(url, encoding= 'unicode_escape')

In [ ]:
df

,Company,rating,reviewer,review_text
0,Minimalix,1,Hardik Ojha,I had a very disappointing experience with Min...
1,Minimalix,5,Vipin Pal,"Recently I subscribed PLP Pro Plus, out of 6 l..."
2,Minimalix,5,Johnsan Barla,As per my overall experience Minimalix is one ...
3,Minimalix,5,The Design Studiokl,1/5\n \n My experience with Minimalix has been...
4,Minimalix,1,Supriya Vimal,"Very BadMost of the leads were either invalid,..."
...,...,...,...,...
994,HomeLane,5,Monu Khan,NaN
995,HomeLane,5,Yadav Madhubala,NaN
996,HomeLane,5,Amrit 2,NaN
997,HomeLane,5,Anish yadav,NaN


In [ ]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Company      999 non-null    object
 1   rating       999 non-null    int64 
 2   reviewer     999 non-null    object
 3   review_text  546 non-null    object
dtypes: int64(1), object(3)
memory usage: 31.3+ KB


(999, 4)

In [ ]:
df.isnull().sum()

,0
Company,0
rating,0
reviewer,0
review_text,453


In [ ]:
df.duplicated().sum()

np.int64(3)

In [ ]:
df["Company"].value_counts()

,count
Company,
Livspace,350
HomeLane,332
Minimalix,317


In [ ]:
df["rating"].describe()

,rating
count,999.000000
mean,4.456456
std,1.283413
min,1.000000
25%,5.000000
50%,5.000000
75%,5.000000
max,5.000000


In [ ]:
df["rating"].value_counts().sort_index()

,count
rating,
1,112
2,5
3,11
4,58
5,813


In [ ]:
df.groupby("Company")["rating"].mean()

,rating
Company,
HomeLane,4.397590
Livspace,4.711429
Minimalix,4.236593


Total reviews me se kitto me text review likha h


In [ ]:
df.groupby("Company")["review_text"].apply(
    lambda x: x.notna().sum()
)

,review_text
Company,
HomeLane,217
Livspace,129
Minimalix,200


Only 54.6% of reviews contain textual feedback
Livspace has the lowest proportion of text reviews despite having the highest rating.

In [ ]:
text_df = df[
    df["review_text"].notna()
].copy()

In [ ]:
text_df.shape

(546, 4)

In [ ]:
text_df["review_length"] = (
    text_df["review_text"]
    .astype(str)
    .apply(len)
)

In [ ]:
text_df.groupby("rating")[
    "review_length"
].mean()

,review_length
rating,
1,441.792208
2,310.333333
3,157.375000
4,174.756098
5,168.880096


Customers who have a poor experience provide significantly more detailed feedback than satisfied customers.

This suggests that dissatisfied customers are more motivated to explain issues, making negative reviews a rich source of operational insights.

Rating	Avg Review Length
1⭐	442 chars
2⭐	310 chars
3⭐	157 chars
4⭐	175 chars
5⭐	169 chars

In [ ]:
text_df["review_length"].describe()

,review_length
count,546.000000
mean,208.417582
std,330.583995
min,4.000000
25%,48.000000
50%,111.500000
75%,222.000000
max,3810.000000


In [ ]:
text_df.nlargest(
    5,
    "review_length"
)[["Company","rating","review_length"]]

,Company,rating,review_length
392,Livspace,5,3810
753,HomeLane,1,2932
835,HomeLane,5,1766
365,Livspace,1,1752
674,HomeLane,1,1609


## Text Preprocessing

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")

In [ ]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
text_df["clean_text"] = (
    text_df["review_text"]
    .apply(clean_text)
)

## Lemmatization


In [ ]:
def lemmatize(text):

    doc = nlp(text)

    return " ".join(
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and len(token.text) > 2
    )

In [ ]:
text_df["processed_text"] = (
    text_df["clean_text"]
    .apply(lemmatize)
)

In [ ]:
text_df[
    [
        "review_text",
        "processed_text"
    ]
].head()

,review_text,processed_text
0,I had a very disappointing experience with Min...,disappointing experience minimalics official j...
1,"Recently I subscribed PLP Pro Plus, out of 6 l...",recently subscribe plp pro plus lead visit con...
2,As per my overall experience Minimalix is one ...,overall experience minimalix good company term...
3,1/5\n \n My experience with Minimalix has been...,experience minimalix extremely disappointing s...
4,"Very BadMost of the leads were either invalid,...",badmost lead invalid badinterested enquiry fee...


In [ ]:
text_df.to_csv(
    "reviews_processed.csv",
    index=False
)

In [ ]:
text_df[
["review_text","processed_text"]
].head(10)

,review_text,processed_text
0,I had a very disappointing experience with Min...,disappointing experience minimalics official j...
1,"Recently I subscribed PLP Pro Plus, out of 6 l...",recently subscribe plp pro plus lead visit con...
2,As per my overall experience Minimalix is one ...,overall experience minimalix good company term...
3,1/5\n \n My experience with Minimalix has been...,experience minimalix extremely disappointing s...
4,"Very BadMost of the leads were either invalid,...",badmost lead invalid badinterested enquiry fee...
5,Excellent experience with minimalix based upon...,excellent experience minimalix base initial co...
6,They provide exceptional leads and engage in e...,provide exceptional lead engage effective comm...
8,I had a very good experience working with Mini...,good experience work minimalix genuinely helpf...
9,Good company with excellent customer service,good company excellent customer service
10,Extremely poor service. The leads shared were ...,extremely poor service lead share interested c...


In [ ]:
text_df[text_df["Company"] == "Minimalix"][
    ["review_text"]
].sample(20, random_state=42)

,review_text
109,Main apki service se bahut impressed hun. Apki...
17,One of the best company in interior design indust
33,I've using Minimalix leads services since 2023...
193,Very bed company aur 3rd grade owner
153,Value for money services
134,"They have given us very good work, their team ..."
76,"Minimalix is fantastic company, Their pre-qual..."
210,Satisfied with Minimalix's interior work.
215,One of the best company for interior design se...
49,This company runs on fake promises. They talk ...


In [ ]:
partner_keywords = [
    "lead",
    "client",
    "subscription",
    "plp",
    "reno",
    "marketing",
    "sales",
    "conversion",
    "enquiry"
]

customer_keywords = [
    "design",
    "interior",
    "home",
    "kitchen",
    "wardrobe",
    "execution",
    "project",
    "designer",
    "installation"
]

In [ ]:
def classify_review(text):

    text = str(text).lower()

    partner_score = sum(
        word in text
        for word in partner_keywords
    )

    customer_score = sum(
        word in text
        for word in customer_keywords
    )

    if partner_score > customer_score:
        return "Partner"

    elif customer_score > partner_score:
        return "Customer"

    else:
        return "General"

In [ ]:
text_df["segment"] = (
    text_df["review_text"]
    .apply(classify_review)
)

## **Customer Behavior Analysis**
General reviews - 267 | Customer reviews - 229 | Partner reviews - 50
Segmentation seems off



In [ ]:
text_df["segment"].value_counts()

,count
segment,
General,267
Customer,229
Partner,50


In [ ]:
text_df[text_df["segment"]=="General"][
    ["Company","review_text"]
].sample(20, random_state=42)

,Company,review_text
169,Minimalix,"Good service, go with 10K packages always...."
101,Minimalix,"Nice company , full supportive"
563,Livspace,Good experience with livspace
791,HomeLane,Gangsara
548,Livspace,"Good service, but regular occurring problem."
719,HomeLane,Nice
129,Minimalix,Good company. Prompt responses from customer s...
931,HomeLane,Hi Homelane is worse company to have a deal wi...
611,Livspace,Thankyou for the quick response. Person was he...
286,Minimalix,Minimliks ek good compny wiswas ke layak hai .


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

In [ ]:
def top_words(company):

    temp = text_df[text_df["Company"] == company]

    vec = CountVectorizer(
        stop_words="english",
        max_features=30
    )

    X = vec.fit_transform(
        temp["processed_text"]
    )

    freq = X.toarray().sum(axis=0)

    words = vec.get_feature_names_out()

    return (
        pd.DataFrame({
            "word": words,
            "freq": freq
        })
        .sort_values(
            "freq",
            ascending=False
        )
    )

In [ ]:
top_words("Minimalix").head(20)

,word,freq
12,minimalix,84
22,service,75
1,company,74
8,good,73
11,lead,70
2,customer,66
10,interior,55
23,support,52
4,design,51
25,team,44


In [ ]:
top_words("HomeLane").head(20)

,word,freq
14,homelane,147
4,design,107
7,experience,94
26,team,90
29,work,87
22,project,71
9,good,55
13,home,54
5,designer,48
15,interior,48


In [ ]:
top_words("Livspace").head(20)

,word,freq
29,work,47
7,good,46
26,team,39
25,service,37
18,livspace,35
6,experience,32
22,project,25
8,great,17
27,thank,17
24,sale,17


## Negative Review Analysis

In [ ]:
negative_df = text_df[text_df["rating"] <= 2]

In [ ]:
negative_df.sample(5)

,Company,rating,reviewer,review_text,review_length,clean_text,processed_text,segment
194,Minimalix,1,Zaid,Ye company ek dam frood hai paise leke fir cal...,216,ye company ek dam frood hai paise leke fir cal...,company dam frood hai paise leke fir bhi nahi ...,General
707,HomeLane,1,Anurag,I booked interior services with homelane aroun...,1157,i booked interior services with homelane aroun...,book interior service homelane mar start discu...,Customer
205,Minimalix,1,Hardik Kaul,Please be careful before choosing to go ahead ...,277,please be careful before choosing to go ahead ...,careful choose ahead hard earn money bad revie...,General
679,HomeLane,1,Rahul Bharadia,"False commitment on timeline, poor budget esti...",953,false commitment on timeline poor budget estim...,false commitment timeline poor budget estimate...,Customer
365,Livspace,1,Devender Arora,A cautionary account for anyone considering Li...,1752,a cautionary account for anyone considering li...,cautionary account consider livspace home inte...,Customer


In [ ]:
negative_df["Company"].value_counts()

,count
Company,
HomeLane,30
Minimalix,29
Livspace,21


In [ ]:
negative_df[
    ["Company","review_text"]
].sample(20, random_state=42)

,Company,review_text
330,Livspace,"Donât take there service, they are unprofess..."
0,Minimalix,I had a very disappointing experience with Min...
193,Minimalix,Very bed company aur 3rd grade owner
331,Livspace,"I paid premium so that educated, experienced t..."
168,Minimalix,Ek dum fraud company. Paise lene k baad respon...
228,Minimalix,Worst company of my life
55,Minimalix,"You trust , when some one makes a commitment a..."
919,HomeLane,Loving how my bedroom interiors turned out. Th...
15,Minimalix,"Thank you for your response. However, my revie..."
72,Minimalix,Title: Poor Value for Money with PLP Pro\n Rev...


| Root Cause  MINIMALIX        | Confidence |
| ---------------------------- | ---------- |  
| Lead Quality Issues          | High       |
| Subscription Value Issues    | High       |
| Post-payment Support         | High       |
| Trust / Credibility Concerns | Medium     |


| Root Cause  HomeLane      | Confidence |
| ------------------------- | ---------- |
| Customization Limitations | High       |
| Service Experience        | High       |
| Project Execution         | Medium     |
| Customer Communication    | Medium     |

| Root Cause Livspace          | Confidence |
| ---------------------------- | ---------- |
| Premium Pricing vs Value Gap | High       |
| Post-Sales Support           | High       |
| Service Quality              | Medium     |
| Expectation Mismatch         | High       |




In [ ]:
negative_df = text_df[text_df["rating"] <= 2].copy()

In [ ]:

def top_negative_words(company):

    temp = negative_df[
        negative_df["Company"] == company
    ]

    vec = CountVectorizer(
        stop_words="english",
        max_features=50
    )

    X = vec.fit_transform(
        temp["processed_text"]
    )

    freq = X.toarray().sum(axis=0)

    return (
        pd.DataFrame({
            "word": vec.get_feature_names_out(),
            "freq": freq
        })
        .sort_values(
            "freq",
            ascending=False
        )
    )

## Root Cause Analysis


In [ ]:
top_negative_words("Minimalix").head(20)

,word,freq
22,lead,30
24,money,20
9,company,14
42,service,10
16,experience,10
23,minimalix,8
15,don,7
45,time,7
18,fraud,7
10,complete,6


RCA

1.  ROI on subscription
2.  Post-payment support
3. Lead quality

In [ ]:
top_negative_words("HomeLane").head(20)

,word,freq
18,homelane,38
9,design,24
32,project,22
41,team,22
49,work,19
39,service,14
14,experience,14
10,designer,13
25,month,11
12,don,10


RCA

1.  Project execution
2.  Timeline management
3. Delivery commitments

This is classic operations-management pain.


In [ ]:
top_negative_words("Livspace").head(20)

,word,freq
48,work,19
41,team,13
34,project,12
2,bad,9
14,don,8
27,livspace,8
30,month,8
6,company,8
39,service,7
17,experience,7


RCA

1.  Service experience
2.  Post sales interaction
3. Value received

# ***Angry customers write 2.6x longer reviews***

In [ ]:
negative_df.nlargest(
    10,
    "review_length"
)[[
    "Company",
    "rating",
    "review_text"
]]

,Company,rating,review_text
753,HomeLane,1,"BEST EXPERIENCE TILL YOU PAY MONEY, WORST AS S..."
365,Livspace,1,A cautionary account for anyone considering Li...
674,HomeLane,1,I recently availed the interior renovation ser...
511,Livspace,1,We are extremely disappointed with our experie...
317,Livspace,1,"""Livpace: Where Deadlines Go to Die""\n Hired L..."
403,Livspace,1,Experience is very worst due to fake and false...
698,HomeLane,1,"I paid â¹25,000 to HomeLane for interior desi..."
707,HomeLane,1,I booked interior services with homelane aroun...
0,Minimalix,1,I had a very disappointing experience with Min...
458,Livspace,1,Stay away from livspace\n Livspace done third ...


| Company   | Main Pain Point                 |
| --------- | ------------------------------- |
| Minimalix | Lead Quality & Subscription ROI |
| HomeLane  | Delays & Project Execution      |
| Livspace  | Post-Sales Service & Value Gap  |


## TF-IDF Insights

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=100
)

X = tfidf.fit_transform(
    negative_df["processed_text"]
)

words = tfidf.get_feature_names_out()

scores = X.sum(axis=0).A1

tfidf_df = pd.DataFrame({
    "word": words,
    "score": scores
}).sort_values(
    "score",
    ascending=False
)

tfidf_df.head(30)

,word,score
15,company,7.926406
47,lead,5.968215
41,homelane,5.727799
82,service,5.683872
4,bad,5.469256
98,work,5.462853
55,money,5.412510
32,experience,5.314633
70,project,5.148554
85,team,5.104193


999 Reviews
3 Companies
546 Text Reviews
453 Rating-only Reviews

## ***Customers care more about experience than design.***

| Root Cause               | Keywords                   | Business Impact          |
| ------------------------ | -------------------------- | ------------------------ |
| Service Quality          | service, support, response | Customer dissatisfaction |
| Value for Money          | money, pay, waste          | Lower trust & referrals  |
| Delivery Delays          | month, time                | Poor customer experience |
| Expectation Gap          | promise, provide           | Trust erosion            |
| Professionalism          | unprofessional, fraud      | Brand damage             |
| Lead Quality (Minimalix) | lead                       | Partner dissatisfaction  |


## Topic Modeling (BERTopic)

In [ ]:
!pip install bertopic
!pip install sentence-transformers
!pip install umap-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.0 MB/s eta 0:00:00


In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(
    min_topic_size=10,
    verbose=True
)

topics, probs = topic_model.fit_transform(
    text_df["processed_text"]
)

2026-06-18 14:03:57,326 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

2026-06-18 14:04:15,472 - BERTopic - Embedding - Completed ✓
2026-06-18 14:04:15,476 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-18 14:04:31,986 - BERTopic - Dimensionality - Completed ✓
2026-06-18 14:04:31,989 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-18 14:04:32,022 - BERTopic - Cluster - Completed ✓
2026-06-18 14:04:32,031 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-18 14:04:32,072 - BERTopic - Representation - Completed ✓


In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,15,-1_ansar_job_good_work,"[ansar, job, good, work, gangsara, akhilesh, a...","[ansar, ansar complete job, ansar good job]"
1,0,244,0_homelane_design_work_team,"[homelane, design, work, team, experience, pro...",[recently avail interior renovation service ho...
2,1,221,1_minimalix_lead_service_customer,"[minimalix, lead, service, customer, good, sup...","[good company excellent customer service, mini..."
3,2,18,2_company_good_nice_job,"[company, good, nice, job, bad, vicky, work, o...","[good company, good company, good company]"
4,3,18,3_painter_job_shahjahan_good,"[painter, job, shahjahan, good, shah, jahan, s...","[shah jahan painter excellent job, shahjahan p..."
5,4,17,4_good_nice_love_excellent,"[good, nice, love, excellent, , , , , , ]","[good, good, good]"
6,5,13,5_post_sale_saroj_service,"[post, sale, saroj, service, sunil, sales, wor...","[post sale service saroj sunil good, post sale..."


In [ ]:
topic_model.get_topic(0)
topic_model.get_topic(1)
topic_model.get_topic(2)

[('company', np.float64(0.7199609157061417)),
 ('good', np.float64(0.5320525668853199)),
 ('nice', np.float64(0.18818251131543376)),
 ('job', np.float64(0.17842575972236613)),
 ('bad', np.float64(0.17842575972236613)),
 ('vicky', np.float64(0.16035500005892148)),
 ('work', np.float64(0.15237052210558433)),
 ('office', np.float64(0.14496805124776693)),
 ('satisfy', np.float64(0.14496805124776693)),
 ('satisfactory', np.float64(0.13597402540063164))]

| Theme ID | Business Theme                           | Count |
| -------- | ---------------------------------------- | ----- |
| 0        | Home Interior Design & Project Execution | 236   |
| 1        | Customer Support & Service Quality       | 99    |
| 2        | Professional Design Experience           | 53    |
| 4        | Lead Quality & Lead Management           | 26    |
| 7        | Subscription Plans & PLP Programs        | 13    |
| 8        | Post-Sales Support Experience            | 13    |
| 9        | Refund, Fraud & Trust Issues             | 11    |


In [ ]:
text_df["topic"] = topics

In [ ]:
text_df[["Company","rating","topic"]].head()

,Company,rating,topic
0,Minimalix,1,1
1,Minimalix,5,1
2,Minimalix,5,1
3,Minimalix,5,1
4,Minimalix,1,1


In [ ]:
pd.crosstab(
    text_df["Company"],
    text_df["topic"]
)

topic,-1,0,1,2,3,4,5
Company,,,,,,,
HomeLane,4,174,31,2,0,6,0
Livspace,9,55,22,4,18,8,13
Minimalix,2,15,168,12,0,3,0


Customers rarely complain about aesthetics alone. Most dissatisfaction originates from service delivery, communication, timelines, support quality, and perceived value.


Sentiment Score + Label

In [ ]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.6 MB/s eta 0:00:00


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

text_df["sentiment_score"] = text_df["review_text"].apply(
    lambda x: analyzer.polarity_scores(str(x))["compound"]
)

def get_sentiment_label(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    return "Neutral"

text_df["sentiment_label"] = text_df["sentiment_score"].apply(
    get_sentiment_label
)

In [ ]:
pd.crosstab(
    text_df["Company"],
    text_df["sentiment_label"],
    normalize="index"
).round(3) * 100

sentiment_label,Negative,Neutral,Positive
Company,,,
HomeLane,14.7,6.0,79.3
Livspace,14.7,12.4,72.9
Minimalix,11.5,7.5,81.0


In [ ]:
topic_map = {
    0:"Design & Project Experience",
    1:"Customer Service & Support",
    2:"Interior Design Professionals",
    3:"Company Reputation",
    4:"Lead Quality",
    5:"General Appreciation",
    6:"Painter Workforce",
    7:"PLP Subscription",
    8:"Post Sales Support",
    9:"Fraud & Refund Concerns"
}

text_df["topic_name"] = text_df["topic"].map(topic_map)

In [ ]:
final_df = text_df[
[
    "Company",
    "rating",
    "reviewer",
    "review_text",
    "review_length",
    "segment",
    "sentiment_score",
    "sentiment_label",
    "topic",
    "topic_name",
    "processed_text"
]]

In [ ]:
final_df = text_df.copy()

final_df.to_csv(
    "voc_analytics_final.csv",
    index=False
)

In [ ]:
final_df.head(5)

,Company,rating,reviewer,review_text,review_length,clean_text,processed_text,segment,topic,sentiment_score,sentiment_label,topic_name
0,Minimalix,1,Hardik Ojha,I had a very disappointing experience with Min...,1054,i had a very disappointing experience with min...,disappointing experience minimalics official j...,Partner,1,-0.9203,Negative,Customer Service & Support
1,Minimalix,5,Vipin Pal,"Recently I subscribed PLP Pro Plus, out of 6 l...",129,recently i subscribed plp pro plus out of lead...,recently subscribe plp pro plus lead visit con...,Partner,1,0.4404,Positive,Customer Service & Support
2,Minimalix,5,Johnsan Barla,As per my overall experience Minimalix is one ...,123,as per my overall experience minimalix is one ...,overall experience minimalix good company term...,General,1,0.7902,Positive,Customer Service & Support
3,Minimalix,5,The Design Studiokl,1/5\n \n My experience with Minimalix has been...,1576,my experience with minimalix has been extremel...,experience minimalix extremely disappointing s...,Partner,1,0.5740,Positive,Customer Service & Support
4,Minimalix,1,Supriya Vimal,"Very BadMost of the leads were either invalid,...",253,very badmost of the leads were either invalid ...,badmost lead invalid badinterested enquiry fee...,Partner,1,-0.4596,Negative,Customer Service & Support


In [ ]:
files.download('voc_analytics_final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

The analysis shows that customer dissatisfaction is driven more by operational and service-related issues than design quality.

Across competitors:

- HomeLane struggles with execution and delivery timelines.
- Livspace faces post-sales service challenges.
- Minimalix faces lead quality and subscription ROI concerns.

Improving communication, service consistency, and lead qualification processes can significantly improve customer satisfaction and retention.